In [1]:
import os
import sys
from pathlib import Path

AGENT_DIR = Path.cwd()
if not (AGENT_DIR / "database.py").exists():
  AGENT_DIR = Path("agent_v0303").resolve()
if str(AGENT_DIR) not in sys.path:
  sys.path.insert(0, str(AGENT_DIR))

os.environ["SONG_ENTRY_DB_PATH"] = str(Path("..") / "data" / "database" / "song_bureaucracy_entries_v0304_run2.db")

AGENT_DIR = Path.cwd()
if not (AGENT_DIR / "database.py").exists():
  AGENT_DIR = Path("agent_v0303").resolve()
if str(AGENT_DIR) not in sys.path:
  sys.path.insert(0, str(AGENT_DIR))

from dotenv import load_dotenv
from llm_client import SimpleLLMClient, LLMTool
load_dotenv()
# 从环境变量读取配置；运行前请确保已设置 OPENROUTER_API_KEY
model_name = os.getenv("OPENROUTER_MODEL", "deepseek/deepseek-v4-flash")
max_tokens = int(os.getenv("OPENROUTER_MAX_TOKENS", "16384"))
llm = SimpleLLMClient(model=model_name, max_tokens=max_tokens)
llm_tool = LLMTool(model_name, llm)
llm_tool.compile()


In [2]:
from database import Database
from config import DICT_DB_PATH, DICT_TABLE, ENTRY_DB_PATH, ensure_save_dir, validate_paths

validate_paths()
SAVE_DIR = ensure_save_dir()
db = Database(str(DICT_DB_PATH), DICT_TABLE, str(ENTRY_DB_PATH))
print(f"Dictionary DB: {DICT_DB_PATH}")
print(f"Entry DB: {ENTRY_DB_PATH}")


Dictionary DB: /Users/zhanyi/Desktop/work/song-bureaucracy/data/database/song_bureaucracy_dictionary.db
Entry DB: /Users/zhanyi/Desktop/work/song-bureaucracy/data/database/song_bureaucracy_entries_v0304.db


In [37]:
# 危险操作：会清空当前结构化结果库。仅在临时测试库上取消注释执行。
# db._recreate_tables()

In [3]:
%reload_ext autoreload
%autoreload 2

import time
import copy
import json
from agent_state import AgentState

start_time = time.time()

dict_index_list = db.get_dictionary_index()
dict_index_text = "\n".join(dict_index_list)
state = AgentState(db=db, dict_index_text=dict_index_text)
todo_dict_entries = dict_index_list[:50]
# print(todo_dict_entries)
# ['河北兵马大元帅府-482']

failed_entries = []

tools_input2facts = {
  "search_dictionary": state.tool_search_dictionary,
  "add_atomic_fact": state.tool_add_atomic_fact,
  "remove_atomic_fact": state.tool_remove_atomic_fact,
  "update_atomic_fact": state.tool_update_atomic_fact,
}

tools_facts2data = {
  "get_entity": state.tool_get_entity,
  "create_entity": state.tool_create_entity,
  "create_timepoint": state.tool_create_timepoint,
  "update_timepoint_attr": state.tool_update_timepoint_attr,
  "create_timepoints_relationship": state.tool_create_timepoints_relationship,
  "append_citation": state.tool_append_citation
}

MAX_llm_loop_count = 10

for i, entry_index in enumerate(todo_dict_entries):
  try:
    print("=" * 40, f"第 {i+1} 个词条 {entry_index}", "=" * 40)
    entry_start_time = time.time()

    records = {
      "input2facts": {},
      "facts2data": {},
    }

    state.prepare_new_round()
    state.append_input_entry(entry_index)
    records["input2facts"]["prompts"] = []

    loop_count = 0
    while not state.finished_facts and loop_count < MAX_llm_loop_count:
      loop_count += 1
      prompt_intput2facts = state.build_prompt_input2facts()
      records["input2facts"]["prompts"].append(prompt_intput2facts)
      print("=" * 30, f"第 {loop_count} 轮", "=" * 30)
      print("=" * 20, "CoT", "=" * 20)
      print(state.cot.get_merged_text())

      response = llm_tool.invoke({"prompt": prompt_intput2facts})
      content_text = response["response"]["response"]["choices"][0]["message"]["content"]

      flag = state.parse_cot(content_text, tools_input2facts)
      state.finished_facts = flag
    
    print("=" * 30, "Result", "=" * 30)
    print("=" * 20, "CoT", "=" * 20)
    print(state.cot.get_merged_text())
    print("=" * 20, "Atomic Facts", "=" * 20)
    print(state.atomic_facts.get_merged_text())

    records["input2facts"]["cot"] = copy.deepcopy(state.cot.chain)
    records["input2facts"]["atomic_facts"] = copy.deepcopy(state.atomic_facts.get_merged_text())

    state.prepare_for_update()
    records["facts2data"]["prompts"] = []

    loop_count = 0
    while not state.finished_update and loop_count < MAX_llm_loop_count:
      loop_count += 1
      prompt_facts2data = state.build_prompt_facts2data()
      records["facts2data"]["prompts"].append(prompt_facts2data)
      print("=" * 30, f"第 {loop_count} 轮", "=" * 30)
      print("=" * 20, "CoT", "=" * 20)
      print(state.cot.get_merged_text())

      response = llm_tool.invoke({"prompt": prompt_facts2data})
      content_text = response["response"]["response"]["choices"][0]["message"]["content"]

      flag = state.parse_cot(content_text, tools_facts2data)
      state.finished_update = flag
    
    print("=" * 30, "Result", "=" * 30)
    print("=" * 20, "CoT", "=" * 20)
    print(state.cot.get_merged_text())
    print("=" * 20, "Related Data Items", "=" * 20)
    print(state.loaded_data_items.get_merged_text())

    records["facts2data"]["cot"] = copy.deepcopy(state.cot.chain)
    records["facts2data"]["loaded_data_items"] = copy.deepcopy(state.loaded_data_items.get_merged_text())

    entry_end_time = time.time()
    print(f"词条 {entry_index} 处理完成，用时 {entry_end_time - entry_start_time} 秒")

    with open(SAVE_DIR / f"records_{i+1}_{entry_index}.json", "w") as f:
      json.dump(records, f, indent=2, ensure_ascii=False)
  except Exception as e:
    print(f"词条 {entry_index} 处理失败，错误信息：{e}")
    failed_entries.append(entry_index)

end_time = time.time()
print(f"所有词条处理完成，用时 {end_time - start_time} 秒")
print(f"处理失败词条 {len(failed_entries)} 个:\n{'\n'.join(failed_entries)}")




======================================== 第 1 个词条 河北兵马大元帅府-482 ========================================
============================== 第 1 轮 ==============================
==================== CoT ====================

============================== 第 2 轮 ==============================
==================== CoT ====================
#    1: assistant's thought
当前已知词条为《河北兵马大元帅府》，位于《辞典》第482页。通读全文后，发现文本中虽多次引用其他文献（如《要录》《宋会要》《金佗粹编》《汪伯彦日历》），但并未出现指向其他《辞典》词条的显式跳转语句（如“详见‘XX’条”、“参见‘XX’”等）。因此，该词条内容自成闭环，无需补充查询。接下来应提取其中涉及的所有原子事实。文本中明确提及的机构为‘河北兵马大元帅府’，并给出了其设立时间（靖康元年十二月一日）、解散时间（建炎元年五月十日），以及多个别称（大元帅府、元帅府、帅府、霸府、天下兵马大元帅府）。此外，还涉及其设立背景（康王赵构被授命为河北兵马大元帅）和职能（募兵勤王抗金，解救京师之围）。根据原子事实提取准则，需将这些信息拆分为多个原子事实，分别对应机构的设立、解散、别称关系等，并保留原始时间点和引用原文片段。

#    2: assistant's action
调用工具：add_atomic_fact(context=引用信息：《辞典》482页 “河北兵马大元帅府” 词条 “基本介绍” 标签
引用原文：北宋靖康元年闰十一月，宋钦宗传檄，授命康王为河北兵马大元帅。十二月一日，赵构开大元帅府，以募兵勤王抗金，解救京师之围为名（《要录》卷1）。
  河北兵马大元帅府
    北宋靖康元年十二月一日
      初置
      职能为募兵勤王抗金，解救京师之围)
成功：新增原子事实: #1
调用工具：add_atomic_fact(context=引用信息：《辞典